In [58]:
import random
# import matplotlib.pyplot as plt
import pandas as pd
# from scipy.spatial.distance import cdist
from ortools.linear_solver import pywraplp
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2


In [47]:
import pandas as pd

raw_data = pd.read_csv('data\distancias_test.csv', index_col=0)

In [48]:
raw_data

,Rosario,El Cairo,Alto Rosario Shopping,Museo Municipal de Bellas Artes Juan B. Castagnino,VIP Rosario,Ruffo Coffee Co.
Rosario,1.0,4.1,3.0,3.5,5.8,2.4
El Cairo,3.2,1.0,5.4,2.9,2.3,3.3
Alto Rosario Shopping,2.4,4.6,1.0,4.9,5.2,2.6
Museo Municipal de Bellas Artes Juan B. Castagnino,3.0,3.4,4.6,1.0,3.8,2.5
VIP Rosario,4.9,1.7,6.1,3.5,1.0,4.5
Ruffo Coffee Co.,2.0,3.3,2.1,2.6,3.9,1.0


In [49]:
for i in raw_data.index:
    raw_data.loc[i, i] = 0

In [50]:
raw_data

,Rosario,El Cairo,Alto Rosario Shopping,Museo Municipal de Bellas Artes Juan B. Castagnino,VIP Rosario,Ruffo Coffee Co.
Rosario,0.0,4.1,3.0,3.5,5.8,2.4
El Cairo,3.2,0.0,5.4,2.9,2.3,3.3
Alto Rosario Shopping,2.4,4.6,0.0,4.9,5.2,2.6
Museo Municipal de Bellas Artes Juan B. Castagnino,3.0,3.4,4.6,0.0,3.8,2.5
VIP Rosario,4.9,1.7,6.1,3.5,0.0,4.5
Ruffo Coffee Co.,2.0,3.3,2.1,2.6,3.9,0.0


In [51]:
def create_data_model(df):
    """Stores the data for the problem."""
    data = {}
    data["distance_matrix"] = df.values.tolist()

    data["num_vehicles"] = 1
    data["depot"] = 0
    return data

In [52]:
data = create_data_model(raw_data)

In [53]:
data

{'distance_matrix': [[0.0, 4.1, 3.0, 3.5, 5.8, 2.4],
  [3.2, 0.0, 5.4, 2.9, 2.3, 3.3],
  [2.4, 4.6, 0.0, 4.9, 5.2, 2.6],
  [3.0, 3.4, 4.6, 0.0, 3.8, 2.5],
  [4.9, 1.7, 6.1, 3.5, 0.0, 4.5],
  [2.0, 3.3, 2.1, 2.6, 3.9, 0.0]],
 'num_vehicles': 1,
 'depot': 0}

In [90]:
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]), data["num_vehicles"], data["depot"]
)
routing = pywrapcp.RoutingModel(manager)

In [91]:
transit_callback_index

1

In [ ]:
def distance_callback(from_index, to_index):
    """Returns the distance between the two nodes."""
    # Convert from routing variable Index to distance matrix NodeIndex.
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node] * 10) # para poder usar el solver de google, necesito ints

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
  

In [93]:
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)


In [94]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

In [97]:
def print_solution(manager, routing, solution):
    """Prints solution on console."""
    print(f"Objective: {solution.ObjectiveValue()/10} miles")
    index = routing.Start(0)
    plan_output = "Route for vehicle 0:\n"
    route_distance = 0
    while not routing.IsEnd(index):
        plan_output += f" {manager.IndexToNode(index)} ->"
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)
    plan_output += f" {manager.IndexToNode(index)}\n"
    plan_output += f"Route distance: {route_distance/10}miles\n"
    print(plan_output)

In [98]:
solution = routing.SolveWithParameters(search_parameters)
if solution:
    print_solution(manager, routing, solution)

Objective: 16.8 miles
Route for vehicle 0:
 0 -> 3 -> 4 -> 1 -> 5 -> 2 -> 0
Route distance: 16.8miles

